In [1]:
import torch
from torch import nn
from kerops.ops.conv import Conv3d

In [2]:
import itertools
from tqdm.notebook import tqdm
from time import perf_counter, sleep
import numpy as np


def mean_std_percentile(x, lo=20, hi=80):
    x = np.asarray(x, dtype=np.float32)

    p_lo, p_hi = np.percentile(x, [lo, hi])

    mask = (x >= p_lo) & (x <= p_hi)
    x_mid = x[mask]

    return x_mid.mean(), x_mid.std()


def bench(func, *args, warmup=25, sleep_ms=100, n_iters=50, q_show=10, quantiles=(20, 80), **specset):
    results = []

    keys = list(specset.keys())
    values = list(specset.values())
    configs = list(itertools.product(*values))

    for config in tqdm(configs, desc="Benchmark configs"):
        if sleep_ms is not None:
            sleep(sleep_ms / 1000)
        kwargs = dict(zip(keys, config))

        try:
            func(*args, **kwargs)
            torch.cuda.synchronize()
        except Exception as e:
            pass

        for _ in range(warmup):
            func(*args, **kwargs)
        torch.cuda.synchronize()

        times_ms = []

        for _ in range(n_iters):
            start = perf_counter()
            func(*args, **kwargs)
            torch.cuda.synchronize()
            end = perf_counter()
            times_ms.append((end - start) * 1e3)

        mean, std = mean_std_percentile(times_ms, *quantiles)
        results.append({
            "spec": kwargs,
            "mean_ms": float(mean),
            "std_ms": float(std),
        })

    best_result = min(results, key=lambda x: x["mean_ms"])
    best_mean = best_result['mean_ms']
    best_std = best_result['std_ms']
    best_spec = best_result['spec']
    print(f'Best spec - {best_mean:.3f}+-{best_std:.3f}ms {best_spec}')

    good_results = []
    for result in results:
        if best_mean * (1 + q_show / 100) >= result["mean_ms"] and result['spec'] != best_spec:
            good_results.append(result)

    if good_results:
        print(30 * '-')
        print("Other good specs:")

        for result in good_results:
            print(f'{result['mean_ms']:.3f}+-{result['std_ms']:.3f}ms {result['spec']}')
    else:
        print(f'Other specs have a time difference of more than {q_show}%')

In [5]:
CIN = 64
COUT = 64
S = 64

x = torch.randn(2, CIN, S, S, S, device='cuda', dtype=torch.float16).to(memory_format=torch.channels_last_3d)
w = torch.randn(3, 3, 3, CIN, COUT, device='cuda', dtype=torch.float16)
conv = nn.Conv3d(CIN, COUT, 3, padding=1, bias=False, device='cuda')

In [10]:
%%timeit -r 5 -n 5
with torch.inference_mode(), torch.amp.autocast('cuda'):
    conv(x)

torch.cuda.synchronize()

1.83 ms ± 65 μs per loop (mean ± std. dev. of 5 runs, 5 loops each)


In [16]:
Conv3d(x, w)
torch.cuda.synchronize()

In [17]:
%%timeit -r 5 -n 5
Conv3d(x, w)
torch.cuda.synchronize()

1.88 ms ± 94.1 μs per loop (mean ± std. dev. of 5 runs, 5 loops each)


In [22]:
%%timeit -r 5 -n 5
Conv3d(x, w, COUT_BLOCK=32, num_warps=2)
torch.cuda.synchronize()

The slowest run took 6430.09 times longer than the fastest. This could mean that an intermediate result is being cached.
2.3 s ± 4.59 s per loop (mean ± std. dev. of 5 runs, 5 loops each)


In [28]:
%%timeit -r 5 -n 5
Conv3d(x, w, COUT_BLOCK=32, num_warps=2)
torch.cuda.synchronize()

1.88 ms ± 59.8 μs per loop (mean ± std. dev. of 5 runs, 5 loops each)


In [6]:
bench(Conv3d, x, w, COUT_BLOCK=[16, 32, 64], num_warps=[1, 2, 4], D_BLOCK=[16, 32], CIN_BLOCK=[16])

Benchmark configs:   0%|          | 0/18 [00:00<?, ?it/s]

Best spec - 1.788+-0.006ms {'COUT_BLOCK': 64, 'num_warps': 4, 'D_BLOCK': 32, 'CIN_BLOCK': 16}
------------------------------
Other good specs:
1.953+-0.057ms {'COUT_BLOCK': 32, 'num_warps': 1, 'D_BLOCK': 32, 'CIN_BLOCK': 16}
1.814+-0.023ms {'COUT_BLOCK': 32, 'num_warps': 2, 'D_BLOCK': 32, 'CIN_BLOCK': 16}
1.846+-0.030ms {'COUT_BLOCK': 64, 'num_warps': 2, 'D_BLOCK': 16, 'CIN_BLOCK': 16}
1.870+-0.015ms {'COUT_BLOCK': 64, 'num_warps': 2, 'D_BLOCK': 32, 'CIN_BLOCK': 16}


In [20]:
%%timeit -r 5 -n 5
Conv3d(x, w, COUT_BLOCK=64, num_warps=4, D_BLOCK=32, CIN_BLOCK=16)
torch.cuda.synchronize()

1.87 ms ± 46 μs per loop (mean ± std. dev. of 5 runs, 5 loops each)


In [21]:
%%timeit -r 5 -n 5
Conv3d(x, w, COUT_BLOCK=32, num_warps=2, D_BLOCK=32, CIN_BLOCK=16)
torch.cuda.synchronize()

1.88 ms ± 77 μs per loop (mean ± std. dev. of 5 runs, 5 loops each)
